In [1]:
# connect to the datatbase 
import mysql.connector

connection = mysql.connector.connect(
    user='root',
    database='depi3'
)
cursor = connection.cursor()

In [2]:
# Create tables for JOIN demonstration

# Create departments table

cursor.execute("""
CREATE TABLE IF NOT EXISTS departments (
    dept_id INT PRIMARY KEY,
    dept_name VARCHAR(50),
    location VARCHAR(50)
)
""")

# Create employees table with department reference
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    emp_id INT PRIMARY KEY,
    emp_name VARCHAR(50),
    salary DECIMAL(10,2),
    dept_id INT,
    hire_date DATE
)
""")

# Create projects table
cursor.execute("""
CREATE TABLE IF NOT EXISTS projects (
    project_id INT PRIMARY KEY,
    project_name VARCHAR(100),
    budget DECIMAL(12,2),
    dept_id INT
)
""")

# Create employee_projects table (many-to-many relationship)
cursor.execute("""
CREATE TABLE IF NOT EXISTS employee_projects (
    emp_id INT,
    project_id INT,
    role VARCHAR(50),
    PRIMARY KEY (emp_id, project_id)
)
""")

In [3]:
# Insert sample data into departments table
departments_data = [
    (1, 'Human Resources', 'New York'),
    (2, 'Engineering', 'San Francisco'),
    (3, 'Marketing', 'Chicago'),
    (4, 'Finance', 'New York'),
    (5, 'Research', 'Boston')
]

cursor.executemany("INSERT IGNORE INTO departments (dept_id, dept_name, location) VALUES (%s, %s, %s)", departments_data)


In [4]:
# Insert sample data into employees table
employees_data = [
    (101, 'Alice Johnson', 75000.00, 1, '2022-01-15'),
    (102, 'Bob Smith', 85000.00, 2, '2021-06-10'),
    (103, 'Charlie Brown', 65000.00, 3, '2023-03-20'),
    (104, 'Diana Prince', 90000.00, 2, '2020-09-05'),
    (105, 'Eve Adams', 70000.00, 4, '2022-11-12'),
    (106, 'Frank Miller', 95000.00, 2, '2019-04-18'),
    (107, 'Grace Lee', 60000.00, 3, '2023-07-01'),
    (108, 'Henry Wilson', 80000.00, 1, '2021-12-03'),
    (109, 'Ivy Chen', 120000.00, None, '2023-01-10')  # Employee without department
]

cursor.executemany("INSERT IGNORE INTO employees (emp_id, emp_name, salary, dept_id, hire_date) VALUES (%s, %s, %s, %s, %s)", employees_data)


In [5]:

# Insert sample data into projects table
projects_data = [
    (201, 'Website Redesign', 150000.00, 2),
    (202, 'HR System Upgrade', 80000.00, 1),
    (203, 'Marketing Campaign', 120000.00, 3),
    (204, 'Budget Analysis Tool', 95000.00, 4),
    (205, 'Mobile App Development', 200000.00, 2),
    (206, 'Data Analytics Platform', 250000.00, 5),
    (207, 'Social Media Strategy', 75000.00, 3),
    (208, 'Legacy System Migration', 300000.00, None)  # Project without assigned department
]

cursor.executemany("INSERT IGNORE INTO projects (project_id, project_name, budget, dept_id) VALUES (%s, %s, %s, %s)", projects_data)


In [6]:
# Insert sample data into employee_projects table
employee_projects_data = [
    (102, 201, 'Lead Developer'),
    (104, 201, 'Senior Developer'),
    (106, 201, 'Technical Lead'),
    (101, 202, 'Project Manager'),
    (108, 202, 'Business Analyst'),
    (103, 203, 'Marketing Specialist'),
    (107, 203, 'Content Creator'),
    (105, 204, 'Financial Analyst'),
    (102, 205, 'Mobile Developer'),
    (104, 205, 'Backend Developer'),
    (109, 206, 'Data Scientist'),
    (103, 207, 'Social Media Manager')
]

cursor.executemany("INSERT IGNORE INTO employee_projects (emp_id, project_id, role) VALUES (%s, %s, %s)", employee_projects_data)


# Aggregate functions

In [8]:
cursor.execute("""
SELECT
    SUM(e.salary),
    MAX(e.salary),
    MIN(e.salary),
    AVG(e.salary)
FROM employees e       
""")

columns = [column[0] for column in cursor.description]
print(columns)
data = cursor.fetchall()
print(data)

['SUM(e.salary)', 'MAX(e.salary)', 'MIN(e.salary)', 'AVG(e.salary)']
[(Decimal('740000.00'), Decimal('120000.00'), Decimal('60000.00'), Decimal('82222.222222'))]


In [ ]:
cursor.execute("""
SELECT COUNT(e.emp_id)
FROM employees e
WHERE e.dept_id = 2
 """)

# this will grab the first element of each tuple → the column’s name/label.
columns = [column[0] for column in cursor.description]
print(columns)
# retrieves all rows returned by the query as a Python list of tuples.
data = cursor.fetchall()
print(data)

['COUNT(e.emp_id)']
[(3,)]


# Group By

In [13]:
cursor.execute("""
SELECT e.dept_id,
COUNT(e.emp_id),
AVG(e.salary)
FROM employees e
WHERE e.dept_id IS NOT NULL
GROUP BY e.dept_id
               
""")
columns = [column[0] for column in cursor.description]
print(columns)
data = cursor.fetchall()
for row in data:
    print(row)

['dept_id', 'COUNT(e.emp_id)', 'AVG(e.salary)']
(1, 2, Decimal('77500.000000'))
(2, 3, Decimal('90000.000000'))
(3, 2, Decimal('62500.000000'))
(4, 1, Decimal('70000.000000'))


# Having + Grouping

In [15]:
cursor.execute("""
               SELECT e.dept_id,
               COUNT(e.emp_id),
               AVG(e.salary)
               FROM employees e
               WHERE e.dept_id IS NOT NULL
               GROUP BY e.dept_id
               HAVING COUNT(e.emp_id) > 2 AND AVG(e.salary) > 50000
 """)

columns = [column[0] for column in cursor.description]
print(columns)
data = cursor.fetchall()
for row in data:
    print(row)

['dept_id', 'COUNT(e.emp_id)', 'AVG(e.salary)']
(2, 3, Decimal('90000.000000'))
